# Environment verification — M0-SETUP-01

Closes the one acceptance criterion of `M0-SETUP-01` that cannot be checked from a
laptop: **`pip install -r requirements.txt` succeeds in a clean Colab cell.**

Run every cell top to bottom on a fresh Colab runtime, then paste the output of
cells 2, 4 and 5 into a new `AGENT_LOG.md` entry.

No project logic lives here — this notebook only reports and calls
(`spec/architecture.md §2`). It is a verification harness, not part of the pipeline.

**What it is actually testing.** The local venvs are empty to start with, so a clean
install there proves very little about Colab, which arrives with its own numpy,
pandas, scipy and sklearn already installed. The real question is whether our lower
bounds force Colab to *downgrade* one of those, or to build something from source
because no wheel matches. Cell 4 watches for exactly that.

In [ ]:
# 2 — what interpreter are we on?
import sys, platform

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import google.colab  # noqa: F401
    print("Runtime : Google Colab")
except ImportError:
    print("Runtime : not Colab (local kernel or other host)")

# spec/architecture.md section 1 pins 3.11. sinter 1.16 needs >= 3.12, so the
# interpreter decides whether we resolve sinter 1.15 or 1.16. Record it.
print()
print("NOTE: record this Python version - it determines the resolved sinter version.")

In [ ]:
# 3 — locate requirements.txt without duplicating its contents anywhere
import pathlib

REQ = None
here = pathlib.Path.cwd()
for parent in [here] + list(here.parents):
    candidate = parent / "requirements.txt"
    if candidate.is_file():
        REQ = candidate.resolve()
        break

if REQ is None:
    print("requirements.txt not reachable from", here)
    print("Upload it now - it is in the repo root.")
    from google.colab import files
    uploaded = files.upload()
    REQ = pathlib.Path(next(iter(uploaded))).resolve()

REPO = REQ.parent
print("Using:", REQ)
print()
print(REQ.read_text(encoding="utf-8"))

In [ ]:
# 4 — the actual acceptance criterion
import subprocess, sys

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQ)],
    capture_output=True,
    text=True,
)
print("pip exit code:", proc.returncode)
print()

# Downgrades of Colab's preinstalled stack, and any from-source build, are the
# two failure modes that would break the one-cell install claim.
WATCH = (
    "Successfully installed",
    "Attempting uninstall",
    "Successfully uninstalled",
    "Building wheel",
    "Running setup.py",
    "ERROR",
    "incompatible",
)
for line in (proc.stdout + proc.stderr).splitlines():
    if any(token in line for token in WATCH):
        print(line)

assert proc.returncode == 0, "pip install FAILED - paste the full output into AGENT_LOG.md"

In [ ]:
# 5 — what actually resolved
import importlib
from importlib.metadata import version, PackageNotFoundError

importlib.invalidate_caches()

PINNED = [
    "numpy", "scipy", "pandas", "pyarrow", "networkx",
    "stim", "sinter", "ldpc",
    "scikit-learn", "lightgbm", "pytest",
    "matplotlib", "pymatching",
]
for name in PINNED:
    try:
        print(name.ljust(14), version(name))
    except PackageNotFoundError:
        print(name.ljust(14), "MISSING")

In [ ]:
# 6 — every import must be warning-free, because pytest.ini sets filterwarnings = error
import subprocess, sys

check = subprocess.run(
    [
        sys.executable, "-W", "error", "-c",
        "import stim, sinter, ldpc, networkx, pandas, numpy, scipy, sklearn, lightgbm, pyarrow; "
        "from ldpc import BpOsdDecoder; "
        "from ldpc.sinter_decoders import SinterBpOsdDecoder; "
        "print('imports clean under -W error')",
    ],
    capture_output=True,
    text=True,
)
print(check.stdout or check.stderr)
assert check.returncode == 0, "an import emitted a warning - pytest would fail on this runtime"

In [ ]:
# 7 — do the pinned CONTRACT decoder parameters work on this runtime?
# Parameters are read from protocol.py, never retyped here (INV-6 / no spec drift).
import sys

src = REPO / "src"
if not src.is_dir():
    print("src/ is not next to requirements.txt - skipping.")
    print("This cell only runs where the repo source is reachable.")
else:
    sys.path.insert(0, str(src))
    import numpy as np
    from ldpc import BpOsdDecoder
    from qecscreen.protocol import DECODER_PARAMS

    kwargs = {k: v for k, v in DECODER_PARAMS.items() if k not in ("library", "decoder")}
    H = np.array([[1, 1, 0], [0, 1, 1]], dtype=np.uint8)
    dec = BpOsdDecoder(H, error_rate=0.01, **kwargs)
    print("BpOsdDecoder accepted:", kwargs)
    print("osd_method as the library stores it:", dec.osd_method)

In [ ]:
# 8 — optional: the full suite, if the repo source is reachable
import subprocess, sys

if not (REPO / "tests").is_dir():
    print("tests/ not reachable - skipping. Cells 4 and 5 are the acceptance criterion.")
else:
    run = subprocess.run(
        [sys.executable, "-m", "pytest"],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(run.stdout[-3000:])
    print(run.stderr[-2000:])
    print("pytest exit code:", run.returncode)

## Recording the result

Paste the output of **cells 2, 4 and 5** into a new `AGENT_LOG.md` entry.

The criterion is met when:

- cell 4 prints `pip exit code: 0`, **and**
- no `Attempting uninstall` line names `numpy`, `pandas`, `scipy` or `scikit-learn`
  — a downgrade of Colab's preinstalled stack means our lower bounds are fighting
  the host, **and**
- no `Building wheel` / `Running setup.py` line appears — that would mean a
  from-source build, which needs a compiler and breaks the one-cell install.

If any of those three fails, that is a real finding: report it rather than
working around it, and do not loosen a bound without recording why.